# Очистка `lectures.csv` 

Пайплайн:

`lectures.csv → STG → Data Quality → ODS/CLEAN → Parquet`

Грейн таблицы: `lecture_id` - одна строка = одна лекция

In [5]:
from pathlib import Path
import duckdb
import pandas as pd

#RAW_DIR = Path("data/raw")
RAW_DIR = Path("RiiidAnswerCorrectnessPrediction")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

def checkpoint(name, ok, detail=""):
    print(("ok " if ok else "not okay") + name + ("" if ok else f" — {detail}"))

print("DuckDB:", duckdb.__version__)
print("RAW_DIR:", RAW_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

DuckDB: 1.5.5
RAW_DIR: /Users/user/Desktop/project/RiiidAnswerCorrectnessPrediction
OUT_DIR: /Users/user/Desktop/project/data/processed


## 1. Пути

In [6]:
LECTURES_PATH = RAW_DIR / "lectures.csv"
LECTURES_CLEAN_PATH = OUT_DIR / "lectures_clean.parquet"
LECTURES_REJECTED_PATH = OUT_DIR / "lectures_rejected.parquet"
LECTURES_DUPLICATES_PATH = OUT_DIR / "lectures_duplicates.parquet"

assert LECTURES_PATH.exists(), f"Не найден файл: {LECTURES_PATH.resolve()}"

LECTURES = sql_path(LECTURES_PATH)
LECTURES_CLEAN = sql_path(LECTURES_CLEAN_PATH)
LECTURES_REJECTED = sql_path(LECTURES_REJECTED_PATH)
LECTURES_DUPLICATES = sql_path(LECTURES_DUPLICATES_PATH)

## 2. STG — сырой слой

In [7]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_lectures AS
SELECT *
FROM read_csv(
    '{LECTURES}',
    header = true,
    all_varchar = true,
    nullstr = ''
)
""")

display(con.sql("SELECT * FROM stg_lectures LIMIT 10").df())

,lecture_id,tag,part,type_of
0,89,159,5,concept
1,100,70,1,concept
2,185,45,6,concept
3,192,79,5,solving question
4,317,156,5,solving question
5,335,114,2,concept
6,484,179,5,concept
7,641,134,6,solving question
8,761,93,1,concept
9,814,80,5,solving question


## 3. ODS — типизация и нормализация

`type_of` приводим к нижнему регистру и убираем лишние пробелы.

In [8]:
con.execute("""
CREATE OR REPLACE TEMP VIEW typed_lectures AS
SELECT
    TRY_CAST(lecture_id AS INTEGER) AS lecture_id,
    TRY_CAST(tag AS INTEGER) AS tag,
    TRY_CAST(part AS SMALLINT) AS part,
    NULLIF(lower(trim(type_of)), '') AS type_of
FROM stg_lectures
""")

display(con.sql("SELECT * FROM typed_lectures LIMIT 10").df())

,lecture_id,tag,part,type_of
0,89,159,5,concept
1,100,70,1,concept
2,185,45,6,concept
3,192,79,5,solving question
4,317,156,5,solving question
5,335,114,2,concept
6,484,179,5,concept
7,641,134,6,solving question
8,761,93,1,concept
9,814,80,5,solving question


## 4. Data Quality

Проверяем:

- Полнота обязательных полей;
- Уникальность `lecture_id`;
- Выбросы (проверяем что поля в нормалных диапазонах): `tag >= 0`, `part ∈ [1,7]`;
- домен `type_of`.


In [9]:
dq = con.sql("""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT lecture_id) AS unique_lecture_ids,

    COUNT(*) FILTER (
        WHERE lecture_id IS NULL
           OR tag IS NULL
           OR part IS NULL
           OR type_of IS NULL
    ) AS missing_mandatory,

    COUNT(*) FILTER (WHERE tag < 0) AS bad_tag,
    COUNT(*) FILTER (WHERE part NOT BETWEEN 1 AND 7) AS bad_part,

    COUNT(*) FILTER (
        WHERE type_of IS NOT NULL
          AND type_of NOT IN (
              'concept',
              'solving question',
              'intention',
              'starter'
          )
    ) AS unexpected_type
FROM typed_lectures
""").df()

display(dq)

display(con.sql("""
SELECT type_of, COUNT(*) AS lectures
FROM typed_lectures
GROUP BY type_of
ORDER BY lectures DESC, type_of
""").df())

r = dq.iloc[0]
dup_rows = int(r["rows_total"] - r["unique_lecture_ids"])

checkpoint("Completeness", int(r["missing_mandatory"]) == 0,
           f"{int(r['missing_mandatory']):,} строк")
checkpoint("Uniqueness lecture_id", dup_rows == 0,
           f"{dup_rows:,} лишних строк")
checkpoint("tag >= 0", int(r["bad_tag"]) == 0,
           f"{int(r['bad_tag']):,} строк")
checkpoint("part ∈ [1,7]", int(r["bad_part"]) == 0,
           f"{int(r['bad_part']):,} строк")

if int(r["unexpected_type"]) == 0:
    print("type_of содержит ожидаемые категории")
else:
    print(f" Неожиданных значений type_of: {int(r['unexpected_type']):,}. "
          "Их не удаляем автоматически — сначала проверьте источник.")

,rows_total,unique_lecture_ids,missing_mandatory,bad_tag,bad_part,unexpected_type
0,418,418,0,0,0,0


,type_of,lectures
0,concept,222
1,solving question,186
2,intention,7
3,starter,3


ok Completeness
ok Uniqueness lecture_id
ok tag >= 0
ok part ∈ [1,7]
type_of содержит ожидаемые категории


## 5. Отделяем технически невалидные строки

In [10]:
invalid_predicate = """
       lecture_id IS NULL
    OR tag IS NULL
    OR part IS NULL
    OR type_of IS NULL
    OR tag < 0
    OR part NOT BETWEEN 1 AND 7
"""

con.execute(f"""
COPY (
    SELECT *
    FROM typed_lectures
    WHERE {invalid_predicate}
)
TO '{LECTURES_REJECTED}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Rejected:",
      con.sql(f"SELECT COUNT(*) FROM read_parquet('{LECTURES_REJECTED}')").fetchone()[0])

Rejected: 0


## 6. Дедупликация и сохранение clean-слоя

In [11]:
con.execute(f"""
COPY (
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY lecture_id
                ORDER BY tag, part, type_of
            ) AS rn
        FROM typed_lectures
        WHERE NOT ({invalid_predicate})
    )
    WHERE rn > 1
)
TO '{LECTURES_DUPLICATES}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        lecture_id,
        tag,
        part,
        type_of
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY lecture_id
                ORDER BY tag, part, type_of
            ) AS rn
        FROM typed_lectures
        WHERE NOT ({invalid_predicate})
    )
    WHERE rn = 1
)
TO '{LECTURES_CLEAN}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Готово:", LECTURES_CLEAN_PATH.resolve())

Готово: /Users/user/Desktop/project/data/processed/lectures_clean.parquet


## 7. Финальный контроль

In [12]:
final_dq = con.sql(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT lecture_id) AS unique_lecture_ids,
    COUNT(*) FILTER (
        WHERE lecture_id IS NULL
           OR tag IS NULL
           OR part NOT BETWEEN 1 AND 7
           OR type_of IS NULL
    ) AS bad_rows
FROM read_parquet('{LECTURES_CLEAN}')
""").df()

display(final_dq)

r = final_dq.iloc[0]
checkpoint("Финал: lecture_id уникален",
           int(r["rows_total"]) == int(r["unique_lecture_ids"]))
checkpoint("Финал: обязательные домены валидны",
           int(r["bad_rows"]) == 0)

,rows_total,unique_lecture_ids,bad_rows
0,418,418,0


ok Финал: lecture_id уникален
ok Финал: обязательные домены валидны


## 8. Результат

- `data/processed/lectures_clean.parquet` — очищенный справочник;
- `data/processed/lectures_rejected.parquet` — технически невалидные строки;
- `data/processed/lectures_duplicates.parquet` — лишние версии повторившихся `lecture_id`.
